# 03. Streaming Delivery & ThinkingConfig Control

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/google-genai/en/03_streaming_and_thinking.ipynb)

Interactive AI applications on **`gemini-3.8-flash`** optimize perceived responsiveness along two independent axes: **streaming delivery (`generate_content_stream`)** cuts Time To First Token (TTFT) by yielding output chunks as soon as generation starts, while **`types.ThinkingConfig(thinking_level=...)`** right-sizes internal reasoning depth (`"low"`, `"medium"`, or `"high"`) to match each task's complexity.
Combining progressive streaming with an explicit thinking level turns multi-second blocking calls into sub-second interactive experiences—reserving deep chain-of-thought compute strictly for tasks that need it.

![Figure 3-1: Blocking Generation vs. Streaming Delivery & Thinking Level Timeline (gemini-3.8-flash)](https://examples.drawlib.com/google-genai/images/03_streaming_and_thinking/ch03_streaming_ttft.png)

*Figure 3-1: Blocking Generation vs. Streaming Delivery & Thinking Level Timeline (gemini-3.8-flash)*

---

## 1. Client Setup & Progressive Streaming (`generate_content_stream`)

Start by setting `MODEL = "gemini-3.8-flash"` and initializing the unified `genai.Client` with your `GEMINI_API_KEY` stored in Google Colab Secrets (**🔑 Secrets** in the left sidebar):

In [ ]:
!pip install -q google-genai

In [ ]:
MODEL = "gemini-3.8-flash"

from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

Whereas `client.models.generate_content()` buffers the entire completion on the server before returning, **`client.models.generate_content_stream()`** yields `GenerateContentResponse` chunks over an HTTP stream as soon as tokens are decoded. Measuring both **`ttft_ms`** (time until the first non-empty `chunk.text` arrives) and **`total_ms`** with `time.perf_counter()` quantifies how quickly users see the response start:

In [ ]:
import time

t0 = time.perf_counter()
ttft_ms = None

for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents="Tell me a short story in 3 sentences.",
):
    if chunk.text:
        if ttft_ms is None:
            ttft_ms = (time.perf_counter() - t0) * 1000
        print(chunk.text, end="", flush=True)

total_ms = (time.perf_counter() - t0) * 1000
print(f"\n\nTTFT: {ttft_ms:.0f} ms | Total: {total_ms:.0f} ms")

> **Note:** Always guard `if chunk.text:` before printing or recording `ttft_ms`. Early or trailing stream frames (such as metadata-only chunks) can carry empty text payloads while updating `usage_metadata`.

---

## 2. Right-Sizing Reasoning with `ThinkingConfig` (`thinking_level`)

`gemini-3.8-flash` is a reasoning-capable model: before emitting visible output tokens, it can spend internal **thinking tokens** (`thoughts_token_count`) planning and verifying its answer. Because thinking runs *before* the first visible output token is emitted, setting `types.ThinkingConfig(thinking_level=...)` directly governs TTFT and token cost across three supported levels on `gemini-3.8-flash`:

- **`thinking_level="low"` (Zero Thoughts / Fastest TTFT)**: Produces `0` internal reasoning tokens on `gemini-3.8-flash`, delivering sub-second TTFT for conversational chat, intent routing, classification, and entity extraction.
- **`thinking_level="medium"` (Default — Balanced Dynamic Reasoning)**: Automatically adapts reasoning depth to prompt complexity—skipping extra thoughts on simple queries while allocating structured reasoning for RAG synthesis, summarization, and SQL drafting.
- **`thinking_level="high"` (Deep Reasoning)**: Dedicates thorough chain-of-thought tokens for multi-step math, algorithm generation, and architectural trade-off analysis.

> **Important (`gemini-3.8-flash` API Rules):** Numeric `thinking_budget` from Gemini 2.5 is deprecated in Gemini 3.8—use `thinking_level` instead (never pass both together). On `gemini-3.8-flash`, only `"low"`, `"medium"`, and `"high"` are valid; passing `"minimal"` or `"xhigh"` returns HTTP `400 INVALID_ARGUMENT`.

![Figure 3-2: ThinkingConfig Levels by Task Complexity (gemini-3.8-flash)](https://examples.drawlib.com/google-genai/images/03_streaming_and_thinking/ch03_thinking_tiers.png)

*Figure 3-2: ThinkingConfig Levels by Task Complexity (gemini-3.8-flash)*

### Benchmarking `thinking_level="low"` vs. `thinking_level="high"`

Let's run the same arithmetic prompt under `thinking_level="low"` and `thinking_level="high"`, capturing `ttft_ms`, `total_ms`, internal `thoughts_token_count`, and visible `candidates_token_count` from `usage_metadata` and printing a formatted comparison table:

In [ ]:
import time
from google.genai import types

prompt = "What is 27 * 43 + 19?"
rows = []

for level in ("low", "high"):
    t0 = time.perf_counter()
    ttft_ms = None
    usage = None
    chunks = []

    for chunk in client.models.generate_content_stream(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level=level),
        ),
    ):
        if chunk.text:
            if ttft_ms is None:
                ttft_ms = (time.perf_counter() - t0) * 1000
            chunks.append(chunk.text)
        if chunk.usage_metadata is not None:
            usage = chunk.usage_metadata

    total_ms = (time.perf_counter() - t0) * 1000
    thoughts_tokens = getattr(usage, "thoughts_token_count", 0) or 0
    output_tokens = getattr(usage, "candidates_token_count", 0) or 0
    rows.append((level, ttft_ms or total_ms, total_ms, thoughts_tokens, output_tokens))
    print(f"[{level}] {''.join(chunks).strip()}")

print(f"\n{'Level':<8} {'TTFT (ms)':>10} {'Total (ms)':>11} {'Thoughts Tok':>14} {'Output Tok':>12}")
print("-" * 59)
for level, ttft_ms, total_ms, thoughts_tokens, output_tokens in rows:
    print(f"{level:<8} {ttft_ms:>10.1f} {total_ms:>11.1f} {thoughts_tokens:>14d} {output_tokens:>12d}")

### Key Takeaways

- **Stream User-Facing Text (`generate_content_stream`)**: Eliminates full-response buffering so users read the opening sentence immediately after prefill and reasoning finish.
- **Match `thinking_level` to Route SLAs**: On `gemini-3.8-flash`, set `thinking_level="low"` (`0` thoughts) on latency-critical routers and chat, keep `"medium"` (default) for general workloads, and reserve `"high"` for complex analytical reasoning.
- **Inspect `thoughts_token_count`**: Monitor `usage_metadata.thoughts_token_count` alongside `candidates_token_count` to track internal reasoning token utilization and latency impact in production.